In [125]:
import pandas as pd
import numpy as np
import json
import random as rnd
import string
import datetime
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
import os

Create Empty Data Frames for All CSVs We want.

In [126]:
last_trigger = {"Content" : datetime.date(2023, 6, 1), "Event": datetime.date(2023, 7, 1)}

In [127]:
#Function to return the probability of an event happening in a given day
def day_prob(pMin, pMax, dMin, dMax):
    #Simulate random day probabilities.
    return 1.0 - (1.0 - rnd.uniform(pMin, pMax)) ** (1.0 / rnd.randint(dMin, dMax))


def generate_random_deals(pMin, pMax, growth):
    #Simulate random partnership/referral deals.
    count = 0
    while rnd.uniform(0, 1) < (rnd.uniform(pMin,pMax) * growth):
        count += 1
    return count


def id_writer(type, number):

    with open("Rules_Tables/rules/tracker_id.json", "r", encoding="utf-8") as file:
        Tracker_ID = json.load(file)
    
    current_ID = int(Tracker_ID[type][-10:])
    new_ids = []

    for id in range(number):
        current_ID += 1
        new_ids.append(f"ID{current_ID:010d}")

    Tracker_ID[type] = f"ID{current_ID:010d}"

    with open("Rules_Tables/rules/tracker_id.json", "w", encoding="utf-8") as file:
        json.dump(Tracker_ID, file, indent = 4)
    
    return new_ids


In [128]:
#Creates a campaign based on the date and the campaign type we need, event or content 

def campaign_creator(day, type):
    event_types = [
        "Trade_show",
        "Conference",
        "Networking_event",
        "Workshop",
        "Product_launche"
    ]

    content_types = [
        "White_Paper",
        "Case_Study",
        "Blog_Post",
        "Guide",
        "Webinar",
        "Infographic"
    ]

    if type == "Event":
        campaign_sub_type = rnd.choice(event_types)
        campaign_type = "Event"
    elif type == "Content":
        campaign_sub_type = rnd.choice(content_types)
        campaign_type = "Content" 
        

    campaign_id = id_writer("Campaign",1)[0]
    campaign_month = day.strftime("%Y-%m")
    campaign_name = f"{campaign_type}_{campaign_month}_{campaign_sub_type}"
    day_str = pd.to_datetime(day).strftime("%Y-%m-%d")


    pd.DataFrame([[
            campaign_id, 
            campaign_name,
            campaign_type, 
            campaign_sub_type,
            day
        ]]).to_csv("Rules_Tables/data/campaign.csv", mode="a", header=False, index=False)
    
    return {
        "CampaignID": campaign_id, 
        "Campaign Name": campaign_name, 
        "Campaign Type": campaign_type,
        "Campaign Sub Type": campaign_sub_type,
        "Campaign Start Date": day_str
        }

In [129]:
CONTENT_DECAY_DAYS = 5
EVENT_MIN_DAYS = 30
EVENT_SCALE = 120

def generate_daily_leads(today, growth):
    #Generate today's new leads by category.
    leads = {
        "Organic": 0, 
        "Content": 0, 
        "Event": 0,
        "Partnership": 0, 
        "Referral": 0}
    
    
    with open("Rules_Tables/rules/current_campaign.json", "r", encoding="utf-8") as file:
        current_campaign = json.load(file)

    last_content_trigger = today - dt.fromisoformat(current_campaign["Content"]["Campaign Start Date"]).date()
    last_event_trigger = today - dt.fromisoformat(current_campaign["Event"]["Campaign Start Date"]).date()
    if today.weekday() < 5:  # Weekday
        #Generate Organic Leads
        leads["Organic"] = int(round(rnd.randint(0, 12) * growth, 0))

        #Check if we had a recent content
        if last_content_trigger.days <= CONTENT_DECAY_DAYS:
            #Generate Content Leads 
            leads["Content"] = int(round(
                (rnd.randint(10, 30) * growth) - (last_content_trigger.days), 0
            ))
        
        elif rnd.uniform(0, 1) <= day_prob(0.25, 0.50, 30, 30):
            leads["Content"] = int(round(rnd.randint(10, 30) * growth, 0))
            current_campaign["Content"] = campaign_creator(today, "Content")


        if (
            rnd.uniform(0, 1)
            < ((last_event_trigger.days / EVENT_SCALE) * growth)
            and last_event_trigger.days > EVENT_MIN_DAYS
            ):
            leads["Event"] = int(round(rnd.randint(0, 200) * growth, 0))
            current_campaign["Event"] = campaign_creator(today,"Event")

        leads["Partnership"] = generate_random_deals(0.05, 0.1, growth)
        leads["Referral"] = generate_random_deals(0.05, 0.1, growth)

    else:  # Weekend
        leads["Organic"] = int(round(rnd.randint(0, 5) * growth, 0))
        if last_content_trigger.days <= CONTENT_DECAY_DAYS:
            leads["Content"] = int(round(rnd.randint(2, 5) * growth, 0))

   
    new_leads = pd.DataFrame(columns = [
                'LeadID','Created Date', 'Lead Status', 'MQL Date',
                'Lead Source', 'Lead Source Details','Rep ID', 'Rep Name',
                'Recent Lead Source', 'Recent Lead Source Details'
                ])
    new_campaign_members = pd.DataFrame(columns =[
                "CampaignMemberID","CampaignID", "LeadID", "Date"
                ])

    for type, value in leads.items(): 
        lead_list =+ value
        lead_ids = id_writer("Lead", value)

        if type in ["Event", "Content"]:

            campaign_member_ID = id_writer ("CampaignMember", value)

            type_leads = pd.DataFrame({
                "LeadID" : lead_ids,
                "Created Date": [today] * value,
                "Lead Status": ['Marketing Qualified'] * value,
                "Lead Source": [type] * value,
                "Recent Lead Source": [type] * value,
                "MQL Date":  [today] * value,
                "Lead Source Details": [current_campaign[type]["Campaign Name"]] * value,
                "Recent Lead Source Details": [current_campaign[type]["Campaign Name"]] * value
            })

            type_campaign_members = pd.DataFrame({
                "CampaignMemberID" : campaign_member_ID,
                "CampaignID": [current_campaign[type]["CampaignID"]] * value,
                "LeadID" : lead_ids,
                "Date": [today] * value
            })
            new_campaign_members = pd.concat([new_campaign_members,type_campaign_members], ignore_index = True)
        else: 
           type_leads = pd.DataFrame({
               "LeadID" :lead_ids,
                "Lead Status": ['Marketing Qualified'] * value,
                "Lead Source": [type] * value,
                "Recent Lead Source": [type] * value,
                "MQL Date":  [today] * value
           })  
        
        new_leads = pd.concat([new_leads,type_leads], ignore_index=True)

    with open("Rules_Tables/rules/current_campaign.json", "w", encoding="utf-8") as file:
        json.dump(current_campaign, file, indent = 4)
    
    return new_leads, new_campaign_members


def daily_converstion(today, leads):
    with open("Rules_Tables/rules/current_campaign.json", "r", encoding="utf-8") as file:
        current_campaign = json.load(file)

    lead_source_rules = pd.read_csv("Rules_Tables/rules/lead_source.csv")
    lead_source_rules['Randomizer'] = lead_source_rules.apply(lambda x: day_prob(x['Min'], x['Max'], 30, 100), axis = 1)

    last_content_trigger = today - dt.fromisoformat(current_campaign["Content"]["Campaign Start Date"]).date()

    working_leads = leads[leads["Lead Status"] != "Qualified"]

    if today.weekday() < 5:  # Weekday
        for index, lead in working_leads.iterrows(): 
            if lead['Lead Status'] == 'New' or lead['Lead Status'] == 'Marketing Qualified':
                if rnd.uniform(0,1)<.75:
                    leads.loc[index,'Lead Status'] = "Qualifying"
                    leads.loc[index,'Qualifying Date'] = today
                
                elif rnd.uniform(0,1)< rnd.uniform(.05,.2):
                    leads.loc[index,'Lead Status'] = "Disqualified"        

            elif lead['Lead Status'] == 'Qualifying':
                if rnd.uniform(0,1) < (lead_source_rules.loc[lead_source_rules['Name'] == lead['Lead Source'], 'Randomizer'].iloc[0]):
                    leads.loc[index,'Lead Status'] = "Qualified"
                    leads.loc[index,'Qualified Date'] = today

                elif ((today-lead['Qualifying Date']).days) > rnd.randint(60,90): 
                    leads.loc[index,'Lead Status'] = "Nurture"
                    leads.loc[index,'Nurture Date'] = today  
            
            elif lead['Lead Status'] == 'Nurture':
                if last_content_trigger.days <= CONTENT_DECAY_DAYS and rnd.uniform(0,1) < day_prob(.05,.1,90,270):
                    leads.loc[index,'Lead Status'] = 'Marketing Qualified'
                    leads.loc[index,'ReMQL Date'] = today
                    leads.loc[index,'Recent Lead Source'] = "Content"

In [130]:
START = datetime.date(2023, 7, 1)
today = datetime.date.today()
growth = 1.0
lead_df = pd.read_csv(
            "Rules_Tables/data/lead.csv",
            parse_dates=["Qualifying Date", "Qualified Date", "Nurture Date", "ReMQL Date", "Created Date", "MQL Date"]
            )
delta = datetime.timedelta(days=1)

for i in range((today - START).days + 1):
    day = START + i * delta
    growth = 1 + ((day-START).days)/900
    new_lead_df, new_campaign_member_df = generate_daily_leads(day, growth)
    new_campaign_member_df.to_csv("Rules_Tables/data/campaign_member.csv", mode="a", index=False, header=False)
    lead_df = pd.concat([lead_df, new_lead_df], ignore_index=True)
    daily_converstion(day, lead_df)

lead_df

,LeadID,Created Date,Lead Status,MQL Date,Qualifying Date,Nurture Date,Qualified Date,Lead Source,Lead Source Details,Rep ID,Rep Name,ReMQL Date,Recent Lead Source,Recent Lead Source Details
0,ID0000000001,NaN,Nurture,2023-07-02,2023-07-05,2023-09-14,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN
1,ID0000000002,NaN,Nurture,2023-07-02,2023-07-04,2023-09-11,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN
2,ID0000000003,NaN,Disqualified,2023-07-02,NaN,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN
3,ID0000000004,NaN,Qualified,2023-07-02,2023-07-03,NaN,2023-09-11,Organic,NaN,NaN,NaN,NaN,Organic,NaN
4,ID0000000005,NaN,Qualified,2023-07-02,2023-07-03,NaN,2023-07-20,Organic,NaN,NaN,NaN,NaN,Organic,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11560,ID0000011561,NaN,Qualifying,2025-09-02,2025-09-02,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN
11561,ID0000011562,NaN,Qualifying,2025-09-02,2025-09-02,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN
11562,ID0000011563,NaN,Qualifying,2025-09-02,2025-09-02,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN
11563,ID0000011564,NaN,Disqualified,2025-09-02,NaN,NaN,NaN,Referral,NaN,NaN,NaN,NaN,Referral,NaN


In [131]:
lead_df[lead_df["Lead Source"]== "Event"].head()

,LeadID,Created Date,Lead Status,MQL Date,Qualifying Date,Nurture Date,Qualified Date,Lead Source,Lead Source Details,Rep ID,Rep Name,ReMQL Date,Recent Lead Source,Recent Lead Source Details
5,ID0000000006,2023-07-03,Nurture,2023-07-03,2023-07-04,2023-09-12,NaN,Event,Event_2023-07_Conference,NaN,NaN,NaN,Event,Event_2023-07_Conference
6,ID0000000007,2023-07-03,Nurture,2023-07-03,2023-07-04,2023-09-08,NaN,Event,Event_2023-07_Conference,NaN,NaN,NaN,Event,Event_2023-07_Conference
7,ID0000000008,2023-07-03,Nurture,2023-07-03,2023-07-04,2023-09-08,NaN,Event,Event_2023-07_Conference,NaN,NaN,NaN,Event,Event_2023-07_Conference
8,ID0000000009,2023-07-03,Qualified,2023-07-03,2023-07-03,NaN,2023-08-01,Event,Event_2023-07_Conference,NaN,NaN,NaN,Event,Event_2023-07_Conference
9,ID0000000010,2023-07-03,Nurture,2023-07-03,2023-07-03,2023-09-08,NaN,Event,Event_2023-07_Conference,NaN,NaN,NaN,Event,Event_2023-07_Conference


In [132]:
lead_df.to_csv("Rules_Tables/data/lead.csv", index=False)